<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_13_exceptions/note_lesson_13_exceptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 13 — Винятки: день, коли каса почала помилятися

В уроці 12 адміністратор запустив `python main.py 2024 липень` і замість звіту отримав `Traceback … ValueError`. А тепер ще й каса віддає чеки **рядками** — `"2024-07-19 18:30;540.00;50;2"` — і частина рядків зіпсована: кома замість крапки, обрізаний рядок, 30 лютого, від'ємна сума.

Сьогодні зробимо так, щоб програма:

1. приймала всі правильні чеки, навіть якщо поруч є зіпсовані;
2. для кожного пропущеного рядка пояснювала, що з ним не так;
3. відповідала адміністраторові людською мовою, а не трасуванням.

Виконуй клітинки **зверху вниз**. Перед клітинками з позначкою **Прогноз** спершу скажи, що буде, і лише потім запускай. Теорія — у книзі: [Урок 13. Винятки](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_13/).

## 🔁 Пригадай (без підглядання)

1. Що станеться при `menu["піца"]`, якщо такого ключа немає? А при `menu.get("піца")`?
2. Що поверне `"2024-07-19 18:30;540.00;50;2".split(";")`?
3. Який тип у `sys.argv[2]` для `python main.py 2024 липень`?

<details>
<summary>Відповіді</summary>

1. `KeyError` зупинить програму; `get` поверне `None`.
2. `['2024-07-19 18:30', '540.00', '50', '2']`.
3. `str` — тому `int("липень")` і зламав програму.

</details>

## 1. Два види помилок

**Прогноз:** розкоментуй другий рядок і запусти. Чи надрукується «Звіт кафе»? Потім закоментуй назад.

In [ ]:
print("Звіт кафе")
# print("Середній чек:", 860.0 / 2

<details>
<summary>Відповідь</summary>

Ні. Незакрита дужка — це `SyntaxError`: Python не може прочитати клітинку і не виконує в ній жодного рядка, навіть правильного першого.

</details>

**Прогноз:** а тут? Розкоментуй другий рядок і запусти.

In [ ]:
print("Звіт кафе")
# print("Середній чек:", 860.0 / 0)
print("Кінець звіту")

<details>
<summary>Відповідь</summary>

«Звіт кафе» надрукується, а далі `ZeroDivisionError`: виняток під час виконання зупиняє програму на рядку, де стався. «Кінець звіту» не надрукується. Трасування читай знизу вгору: тип і повідомлення → рядок, що впав → хто його викликав.

</details>

## 2. Каса віддає рядки

Дев'ять рядків за день. `RawOrder` — той самий чек, що в уроці 12.

In [ ]:
from datetime import datetime
from typing import NamedTuple


class RawOrder(NamedTuple):
    total_bill: float
    tip: float
    size: int
    timestamp: datetime


KASA_LINES = [
    "2024-07-19 18:30;540.00;50;2",
    "2024-07-19 12:10;320.00;30;1",
    "2024-07-19 19:05;540,00;40;3",
    "2024-07-20 20:15;980.00;120;4",
    "2024-07-20 13:40;760.00",
    "2024-02-30 19:00;450.00;0;5",
    "2024-07-21 18:00;-120.00;0;2",
    "2024-07-21 14:20;610.00;60;0",
    "2024-07-21 21:30;1200.00;150;6",
]

In [ ]:
def parse_line(line):
    """Рядок каси -> RawOrder."""
    time_text, bill_text, tip_text, size_text = line.split(";")
    timestamp = datetime.strptime(time_text, "%Y-%m-%d %H:%M")
    return RawOrder(float(bill_text), float(tip_text), int(size_text), timestamp)


def load_orders(lines):
    orders = []
    for line in lines:
        orders.append(parse_line(line))
    return orders


print(parse_line(KASA_LINES[0]))

**Прогноз:** скільки чеків розбере `load_orders(KASA_LINES)`?

Клітинка нижче впаде — так і задумано. Прочитай трасування знизу вгору і знайди рядок каси, що зламав розбір.

In [ ]:
# orders = load_orders(KASA_LINES)   # розкоментуй, запусти, прочитай трасування, закоментуй назад

<details>
<summary>Відповідь</summary>

Жодного: третій рядок (`540,00` з комою) дає `ValueError: could not convert string to float` і зупиняє весь розбір. Виняток виник у `float()` всередині `parse_line`, піднявся в `load_orders` і вище.

</details>

## 3. `try` / `except`

`try` — код, що може впасти; `except ValueError` — що робити, якщо впав саме так. `as error` дає повідомлення Python.

In [ ]:
def parse_bill(text):
    try:
        return float(text)
    except ValueError:
        print("Не число:", text)
        return None


print(parse_bill("540.00"))
print(parse_bill("540,00"))


try:
    float("540,00")
except ValueError as error:
    print("Каса надіслала не число:", error)

### `else` і `finally`

**Прогноз:** які рядки надрукує `check_line` для першого рядка каси і для третього (з комою)?

In [ ]:
def check_line(line):
    print("try: розбираю", line[:16])
    try:
        order = parse_line(line)
        print("try: розібрано")
    except ValueError as error:
        print("except:", error)
    else:
        print("else: чек на", order.total_bill)
    finally:
        print("finally: рядок перевірено")


check_line(KASA_LINES[0])
print("---")
check_line(KASA_LINES[2])

<details>
<summary>Відповідь</summary>

Для першого: `try`, `try: розібрано`, `else`, `finally`. Для третього: `try`, `except`, `finally` — `try` перервався на `parse_line(line)`, `else` пропущено, `finally` виконується завжди.

</details>

## 4. Ієрархія і кілька `except`

`except` ловить свій тип **і всіх нащадків**: `calendar.IllegalMonthError` — різновид `ValueError`. Кілька `except` перевіряються згори вниз, спрацьовує перший, що підійшов.

In [ ]:
import calendar

try:
    calendar.monthrange(2024, 13)
except ValueError as error:
    print("Не той місяць:", error)


menu = {"борщ": 95, "вареники": 80, "вода": 0}


def price_per_guest(dish, guests):
    try:
        return menu[dish] / guests
    except KeyError:
        return f"страви {dish} немає в меню"
    except ZeroDivisionError:
        return "гостей має бути хоча б один"


print(price_per_guest("борщ", 2))
print(price_per_guest("піца", 2))
print(price_per_guest("борщ", 0))

## 5. `raise`: правила кафе

`-120.00` і `0` гостей Python розбере без помилок — але для кафе такі чеки неможливі. Перевіряємо самі й повідомляємо так само, як Python: винятком. Ця версія `parse_line` замінює наївну.

In [ ]:
def parse_line(line):
    """Рядок каси -> RawOrder. Зіпсований рядок -> ValueError з поясненням."""
    fields = line.split(";")
    if len(fields) != 4:
        raise ValueError(f"очікували 4 поля, а маємо {len(fields)}")
    time_text, bill_text, tip_text, size_text = fields
    timestamp = datetime.strptime(time_text, "%Y-%m-%d %H:%M")
    bill, tip, size = float(bill_text), float(tip_text), int(size_text)
    if bill <= 0:
        raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
    if tip < 0:
        raise ValueError(f"чайові не можуть бути від'ємними: {tip}")
    if size < 1:
        raise ValueError(f"гостей має бути хоча б один, а маємо {size}")
    return RawOrder(bill, tip, size, timestamp)


print(parse_line("2024-07-21 21:30;1200.00;150;6").size)
# parse_line("2024-07-21 18:00;-120.00;0;2")   # розкоментуй: ValueError з нашим поясненням

## 6. Де ловити: там, де знаєш, що відповісти

`parse_line` бачить лише один рядок. `load_orders` знає номер рядка і що робити з поганим: пропустити й записати причину. Тож `try` — тут.

**Прогноз:** скільки чеків буде прийнято і скільки пояснень?

In [ ]:
def load_orders(lines):
    """Правильні чеки і список пояснень до пропущених рядків."""
    orders = []
    errors = []
    for number, line in enumerate(lines, start=1):
        try:
            orders.append(parse_line(line))
        except ValueError as error:
            errors.append(f"рядок {number}: {error}")
    return orders, errors


orders, errors = load_orders(KASA_LINES)
print("Прийнято чеків:", len(orders))
for message in errors:
    print(message)

<details>
<summary>Відповідь</summary>

4 чеки і 5 пояснень: кома (рядок 3), обрізаний рядок (5), 30 лютого (6), від'ємна сума (7), 0 гостей (8). Помилки не минули тихо — Errors should never pass silently.

</details>

## 7. Антипатерн «підгузок»

**Прогноз:** рядок каси правильний. Що надрукує клітинка?

In [ ]:
accepted = []
try:
    order = parse_line(KASA_LINES[0])
    accepted.apend(order)
except:
    print("Зіпсований рядок каси")

<details>
<summary>Відповідь</summary>

«Зіпсований рядок каси» — хоча рядок правильний. Впала друкарська помилка `apend` (`AttributeError`), а голий `except:` сховав її і звинуватив касу. Заміни `except:` на `except ValueError:` і запусти ще раз: тепер `AttributeError` чесно покаже справжню причину. Потім виправ `apend` на `append`.

</details>

## 8. EAFP чи LBYL

**Прогноз:** для яких рядків `isdigit()` і `int()` не погодяться?

In [ ]:
for text in ["7", " 7", "-5", "07"]:
    print(repr(text), text.isdigit(), end=" ")
    try:
        print(int(text))
    except ValueError:
        print("ValueError")

<details>
<summary>Відповідь</summary>

`" 7"` і `"-5"`: `isdigit()` каже `False`, а `int()` їх перетворює. Тому для перетворень — EAFP: `try: int(text)`. LBYL (`in`, `get`) — коли відсутність нормальна, а не помилка.

</details>

## 9. Розібраний приклад: зрозумілі помилки для адміністратора

`parse_period` піднімає `ValueError` з поясненням, `main` ловить і друкує пояснення та підказку. `raise` всередині `except` замінює технічне повідомлення `int()` людським.

In [ ]:
def parse_period(args):
    """['main.py', '2024', '7'] -> (2024, 7). Некоректні аргументи -> ValueError."""
    if len(args) != 3:
        raise ValueError("потрібно два аргументи: рік і місяць")
    try:
        year, month = int(args[1]), int(args[2])
    except ValueError:
        raise ValueError(f"рік і місяць мають бути числами, а маємо {args[1]} і {args[2]}")
    if not 1 <= month <= 12:
        raise ValueError(f"місяць має бути від 1 до 12, а маємо {month}")
    return year, month


def main(args):
    try:
        year, month = parse_period(args)
    except ValueError as error:
        print("Помилка:", error)
        print("Використання: python main.py РІК МІСЯЦЬ, наприклад: python main.py 2024 7")
        return 1
    print(f"Звіт кафе за {month:02d}.{year}")
    return 0


main(["main.py", "2024", "липень"])
main(["main.py", "2024", "13"])
main(["main.py", "2024"])
main(["main.py", "2024", "7"])

## 🛠 Вправа 1. Питати місяць, доки не введуть правильно

Напиши `ask_month(ask)`: викликає `ask("Місяць (1–12): ")`, при нечисловій відповіді друкує `Помилка: потрібне число, а маємо …`, при числі поза 1–12 — `Помилка: місяць має бути від 1 до 12`, і питає знову. Повертає перший правильний місяць.

`fake_input` замінює клавіатуру відповідями зі списку. З клавіатурою: `ask_month(input)`.

In [ ]:
def fake_input(answers):
    """Функція, що замість клавіатури повертає відповіді зі списку по черзі."""
    it = iter(answers)

    def ask(prompt):
        answer = next(it)
        print(prompt + answer)
        return answer

    return ask

In [ ]:
def ask_month(ask):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    while True:
        text = ask("Місяць (1–12): ")
        try:
            month = int(text)
        except ValueError:
            print("Помилка: потрібне число, а маємо", text)
            continue
        if 1 <= month <= 12:
            return month
        print("Помилка: місяць має бути від 1 до 12")
    # END SOLUTION


month = ask_month(fake_input(["липень", "13", " 7 "]))
print(month)
assert month == 7
assert ask_month(fake_input(["12"])) == 12
print("✅ Вправа 1 пройдена")

## 🛠 Вправа 2. Меню і бюджет

`portions_for_budget(dish, budget, menu)` — скільки порцій можна взяти на бюджет (`budget // ціна`):

- страви немає → зловити `KeyError` і підняти `ValueError("страви … немає в меню")`;
- безкоштовна страва → зловити `ZeroDivisionError` і повернути `None`;
- від'ємний бюджет → власний `raise ValueError("бюджет не може бути від'ємним")` до обчислень;
- жодного голого `except:` і жодної перевірки `if dish in menu`.

In [ ]:
menu = {"борщ": 95, "вареники": 80, "узвар": 35, "вода": 0}


def portions_for_budget(dish, budget, menu):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    if budget < 0:
        raise ValueError("бюджет не може бути від'ємним")
    try:
        return budget // menu[dish]
    except KeyError:
        raise ValueError(f"страви {dish} немає в меню")
    except ZeroDivisionError:
        return None
    # END SOLUTION


assert portions_for_budget("борщ", 300, menu) == 3
assert portions_for_budget("узвар", 100, menu) == 2
assert portions_for_budget("вода", 100, menu) is None
for dish, budget, message in [("піца", 300, "страви піца немає в меню"),
                              ("борщ", -50, "бюджет не може бути від'ємним")]:
    try:
        portions_for_budget(dish, budget, menu)
    except ValueError as error:
        assert str(error) == message, error
    else:
        raise AssertionError(f"очікували ValueError для {dish}, {budget}")
print("✅ Вправа 2 пройдена")

## 🛠 Вправа 3. Знайди помилку

У кожній клітинці обробка винятків зроблена неправильно. Запусти, поясни, що сталося, і виправ так, щоб `assert` пройшов.

In [ ]:
# Баг 1: правильний рядок каси — а повідомлення каже, що зіпсований
accepted = []
message = "прийнято"
try:
    accepted.apend(parse_line(KASA_LINES[0]))
except:
    message = "зіпсований рядок"
# BEGIN SOLUTION
accepted = []
message = "прийнято"
try:
    accepted.append(parse_line(KASA_LINES[0]))
except ValueError:
    message = "зіпсований рядок"
# END SOLUTION
assert message == "прийнято" and len(accepted) == 1
print("✅ Баг 1 виправлено")

In [ ]:
# Баг 2: розкоментуй рядки нижче — чому програма падає, хоча є except?
# try:
#     price = menu["піца"]
# except IndexError:
#     price = None
# BEGIN SOLUTION
try:
    price = menu["піца"]
except KeyError:
    price = None
# END SOLUTION
assert price is None
print("✅ Баг 2 виправлено")

In [ ]:
# Баг 3: чому ніколи не з'являється «Рік має бути числом»?
def year_message(text):
    try:
        int(text)
    except Exception:
        return "Щось пішло не так"
    except ValueError:
        return "Рік має бути числом"
    return "ок"
# BEGIN SOLUTION
def year_message(text):
    try:
        int(text)
    except ValueError:
        return "Рік має бути числом"
    return "ок"
# END SOLUTION


assert year_message("двадцять") == "Рік має бути числом"
assert year_message("2024") == "ок"
print("✅ Баг 3 виправлено")

<details>
<summary>Пояснення</summary>

1. Голий `except:` ховав `AttributeError` від `apend`. 2. Словник піднімає `KeyError`, а не `IndexError`. 3. `except Exception` стоїть першим і забирає `ValueError`: вузькі типи — вище, широкі — нижче або зовсім не потрібні.

</details>

## ✅ Самоперевірка

1. Що виконається з програми, у третьому рядку якої `SyntaxError`? А якщо там `ZeroDivisionError`?
2. У `try` три рядки, виняток виник у першому. Чи виконаються другий і третій?
3. Чим відрізняються `else` і `finally`?
4. Чи спрацює `except ValueError:` для `calendar.monthrange(2024, 13)`?
5. Чому `parse_line` піднімає `ValueError`, а не друкує і повертає `None`?
6. Чому `try` стоїть у `load_orders`, а не в `parse_line`?

<details>
<summary>Відповіді</summary>

1. `SyntaxError` — нічого; `ZeroDivisionError` — перші два рядки.
2. Ні: виконання переходить до `except`.
3. `else` — лише якщо винятку не було; `finally` — завжди.
4. Так: `IllegalMonthError` — нащадок `ValueError`.
5. `None` легко не помітити, і він зламає звіт пізніше; виняток або обробить той, хто знає, що робити, або зупинить програму з точним повідомленням.
6. `load_orders` знає номер рядка і що робити з поганим рядком.

</details>

### Шпаргалка

```python
try:
    order = parse_line(line)          # лише те, що може впасти
except ValueError as error:           # конкретний тип, не голий except:
    errors.append(f"рядок {n}: {error}")
else:
    orders.append(order)              # лише якщо винятку не було
finally:
    checked += 1                      # завжди

except (KeyError, ZeroDivisionError): # кілька типів з однаковою обробкою
raise ValueError("пояснення")         # правило програми порушено

# трасування — знизу вгору: тип і повідомлення → рядок → хто викликав
# EAFP: try: int(text)       LBYL: menu.get(dish), dish in menu
```

## Далі

**Урок 14 — Файли, менеджери контексту та JSON.** Рядки каси прийдуть з файлу: з'являться `FileNotFoundError` і `with`, який закриває файл навіть тоді, коли всередині стався виняток.

Довідник з усією темою — [`notes_exceptions.ipynb`](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_13_exceptions/notes_exceptions.ipynb).